# 04 — Raw Data Exploration

**Goal:** Resolve the specific open questions left by 02 (Data Quality) and 03 (Data
Profiling) by looking at actual row-level values — not another metadata pass.

This notebook is deliberately narrow. It does **not** explore all 369 raw tables. It
answers four specific questions that 05's cleaning plan needs settled before writing
dbt staging models:

1. Are the 3 broken `_areacodes` tables exact duplicates, or near-duplicates?
2. What do `trade_matrix`'s wide year/flag columns actually look like, row by row?
3. Why does `commodity_prices` have 2485 correlation pairs?
4. Spot-check a couple of the 90-table profiling shortlist for real values.

Each section queries `raw.*` directly.

In [11]:
from _bootstrap import project_root
from src.audit.run_management import get_latest_run_id, list_run_ids

import json
import polars as pl

pl.Config.set_tbl_cols(-1)          # show all columns, no collapsing
pl.Config.set_tbl_width_chars(200)  # widen the rendered table
pl.Config.set_fmt_str_lengths(120)  # don't truncate long strings
pl.Config.set_tbl_rows(50)          # show more rows before truncating vertically

from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(True)
print("Connected")

Connected


## 1. Are the 3 broken `_areacodes` tables exact duplicates or near-duplicates?

From 02: `foodbalancesheets_areacodes`, `foodbalancesheetshistoric_areacodes`, and
`forestry_trade_flows_areacodes` each have ~50% duplicate rows. If they're exact
duplicates, `DISTINCT` in staging is sufficient. If they're near-duplicates (e.g. a
trailing space or case difference), a normalize-then-dedupe step is needed instead.

In [12]:
areacodes_tables_to_check = [
    "foodbalancesheets_areacodes",
    "foodbalancesheetshistoric_areacodes",
    "forestry_trade_flows_areacodes",
]

# NOTE: "SELECT *, ROW_NUMBER() OVER (...)" fails in DuckDB — a bare "*" can only be the
# root expression in a SELECT list, not mixed with other expressions. Use GROUP BY ALL
# to count distinct rows instead, which sidesteps the restriction entirely.
for table in areacodes_tables_to_check:
    print(f"--- {table} ---")
    counts = conn.execute(f"""
        SELECT
            (SELECT count(*) FROM raw.{table}) AS total_rows,
            (SELECT count(*) FROM (SELECT * FROM raw.{table} GROUP BY ALL)) AS distinct_rows_exact
    """).fetchone()
    total_rows, distinct_rows_exact = counts
    print(f"total_rows={total_rows}, distinct_rows_exact={distinct_rows_exact}, "
          f"exact_duplicate_rows={total_rows - distinct_rows_exact}")

--- foodbalancesheets_areacodes ---
total_rows=426, distinct_rows_exact=213, exact_duplicate_rows=213
--- foodbalancesheetshistoric_areacodes ---
total_rows=434, distinct_rows_exact=217, exact_duplicate_rows=217
--- forestry_trade_flows_areacodes ---
total_rows=423, distinct_rows_exact=216, exact_duplicate_rows=207


### Look at one actual duplicated row per table

If the exact-duplicate count above matches what 02's `dq_reports.duplicate_rows` found,
they're exact duplicates — confirm by eyeballing one repeated value below.

In [13]:
for table in areacodes_tables_to_check:
    print(f"--- {table}: one value that appears more than once ---")
    dupe_example = conn.execute(f"""
        SELECT *, count(*) AS n
        FROM raw.{table}
        GROUP BY ALL
        HAVING count(*) > 1
        LIMIT 3
    """).pl()
    print(dupe_example)

--- foodbalancesheets_areacodes: one value that appears more than once ---
shape: (3, 4)
┌────────────┬───────────┬──────────┬─────┐
│  Area Code ┆  M49 Code ┆  Area    ┆ n   │
│ ---        ┆ ---       ┆ ---      ┆ --- │
│ i64        ┆ str       ┆ str      ┆ i64 │
╞════════════╪═══════════╪══════════╪═════╡
│ 5200       ┆ '019      ┆ Americas ┆ 2   │
│ 13         ┆ '048      ┆ Bahrain  ┆ 2   │
│ 115        ┆ '116      ┆ Cambodia ┆ 2   │
└────────────┴───────────┴──────────┴─────┘
--- foodbalancesheetshistoric_areacodes: one value that appears more than once ---
shape: (3, 4)
┌───────────┬──────────┬───────────┬─────┐
│ Area Code ┆ M49 Code ┆ Area      ┆ n   │
│ ---       ┆ ---      ┆ ---       ┆ --- │
│ i64       ┆ str      ┆ str       ┆ i64 │
╞═══════════╪══════════╪═══════════╪═════╡
│ 3         ┆ '008     ┆ Albania   ┆ 2   │
│ 5300      ┆ '142     ┆ Asia      ┆ 2   │
│ 5206      ┆ '029     ┆ Caribbean ┆ 2   │
└───────────┴──────────┴───────────┴─────┘
--- forestry_trade_flows_areaco

## 2. What do `trade_matrix`'s wide year/flag columns actually contain?

From 03: `trade_matrix` has 78 wide year columns (`Y1986`/`Y1986F` ... `Y2024`/`Y2024F`)
and needs unpivoting to long format in staging. Before writing that dbt model, confirm:
what values do the `F` (flag) columns actually hold, and are there rows where every
single year is null (candidates to drop before unpivoting, to avoid a flood of
all-null long-format rows)?

In [14]:
trade_matrix_sample = conn.execute("""
    SELECT "Reporter Countries", "Partner Countries", "Item", "Element", "Unit",
           "Y2020", "Y2020F", "Y2021", "Y2021F", "Y2022", "Y2022F"
    FROM raw.trade_matrix
    LIMIT 10
""").pl()
trade_matrix_sample

Reporter Countries,Partner Countries,Item,Element,Unit,Y2020,Y2020F,Y2021,Y2021F,Y2022,Y2022F
str,str,str,str,str,f64,str,f64,str,str,str
"""Afghanistan""","""Argentina""","""Cake, oilseeds nes""","""Import quantity""","""t""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Cake, oilseeds nes""","""Import value""","""1000 USD""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Food preparations n.e.c.""","""Import quantity""","""t""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Food preparations n.e.c.""","""Import value""","""1000 USD""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Meat of chickens, fresh or chilled""","""Import quantity""","""t""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Meat of chickens, fresh or chilled""","""Import value""","""1000 USD""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Refined sugar""","""Import quantity""","""t""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Refined sugar""","""Import value""","""1000 USD""",null,null,null,null,null,null
"""Afghanistan""","""Argentina""","""Sweet corn, frozen""","""Import quantity""","""t""",null,null,null,null,null,null


In [15]:
# What distinct values do flag columns actually take? Check a few years.
for flag_col in ["Y2020F", "Y2021F", "Y2022F"]:
    distinct_flags = conn.execute(f'SELECT DISTINCT "{flag_col}" FROM raw.trade_matrix ORDER BY 1').pl()
    print(f"{flag_col}: {distinct_flags[flag_col].to_list()}")

Y2020F: ['A', 'E', 'I', 'X', None]
Y2021F: ['A', 'E', 'I', 'X', None]
Y2022F: ['A', 'E', 'I', 'X', None]


In [16]:
# Rows where every year value (1986-2024) is null — candidates to drop before unpivoting.
year_cols = [f"Y{y}" for y in range(1986, 2025)]
null_check = " AND ".join(f'"{c}" IS NULL' for c in year_cols)

all_null_years = conn.execute(f"""
    SELECT count(*) AS rows_with_every_year_null
    FROM raw.trade_matrix
    WHERE {null_check}
""").fetchone()[0]

total_rows = conn.execute("SELECT count(*) FROM raw.trade_matrix").fetchone()[0]
print(f"Rows with every year null: {all_null_years} / {total_rows} "
      f"({all_null_years / total_rows * 100:.2f}%)")

Rows with every year null: 0 / 6640547 (0.00%)


## 3. Why does `commodity_prices` have 2485 correlation pairs?

From 03: higher correlation count than `trade_matrix` (465), but `wide_year_column_count
= 0` — the wide-year explanation doesn't apply. Check actual column names and a sample
of rows to see whether it's wide by commodity/price-series, or genuinely redundant
unit-conversion columns.

In [17]:
commodity_prices_columns = conn.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'raw' AND table_name = 'commodity_prices'
    ORDER BY ordinal_position
""").pl()
print(f"{commodity_prices_columns.height} columns")
commodity_prices_columns

72 columns


column_name,data_type
str,str
"""Date""","""VARCHAR"""
"""Crude oil, average ($/bbl)""","""DOUBLE"""
"""Crude oil, Brent ($/bbl)""","""DOUBLE"""
"""Crude oil, Dubai ($/bbl)""","""DOUBLE"""
"""Crude oil, WTI ($/bbl)""","""DOUBLE"""
"""Coal, Australian ($/mt)""","""DOUBLE"""
"""Coal, South African ** ($/mt)""","""DOUBLE"""
"""Natural gas, US ($/mmbtu)""","""DOUBLE"""
"""Natural gas, Europe ($/mmbtu)""","""DOUBLE"""


In [18]:
commodity_prices_sample = conn.execute("SELECT * FROM raw.commodity_prices LIMIT 5").pl()
commodity_prices_sample

Date,"Crude oil, average ($/bbl)","Crude oil, Brent ($/bbl)","Crude oil, Dubai ($/bbl)","Crude oil, WTI ($/bbl)","Coal, Australian ($/mt)","Coal, South African ** ($/mt)","Natural gas, US ($/mmbtu)","Natural gas, Europe ($/mmbtu)","Liquefied natural gas, Japan ($/mmbtu)",Natural gas index (2010=100),Cocoa ($/kg),"Coffee, Arabica ($/kg)","Coffee, Robusta ($/kg)","Tea, avg 3 auctions ($/kg)","Tea, Colombo ($/kg)","Tea, Kolkata ($/kg)","Tea, Mombasa ($/kg)",Coconut oil ($/mt),Groundnuts ($/mt),Fish meal ($/mt),Groundnut oil ** ($/mt),Palm oil ($/mt),Palm kernel oil ($/mt),Soybeans ($/mt),Soybean oil ($/mt),Soybean meal ($/mt),Rapeseed oil ($/mt),Sunflower oil ($/mt),Barley ($/mt),Maize ($/mt),Sorghum ($/mt),"Rice, Thai 5% ($/mt)","Rice, Thai 25% ($/mt)","Rice, Thai A.1 ($/mt)","Rice, Viet Namese 5% ($/mt)","Wheat, US SRW ($/mt)","Wheat, US HRW ($/mt)","Banana, Europe ($/kg)","Banana, US ($/kg)",Orange ($/kg),Beef ** ($/kg),Chicken ** ($/kg),Lamb ** ($/kg),"Shrimps, Mexican ($/kg)","Sugar, EU ($/kg)","Sugar, US ($/kg)","Sugar, world ($/kg)","Tobacco, US import u.v. ($/mt)","Logs, Cameroon ($/cubic meter)","Logs, Malaysian ($/cubic meter)","Sawnwood, Cameroon ($/cubic meter)","Sawnwood, Malaysian ($/cubic meter)",Plywood (cents/sheet),"Cotton, A Index ($/kg)","Rubber, TSR20 ** ($/kg)","Rubber, RSS3 ($/kg)",Phosphate rock ($/mt),DAP ($/mt),TSP ($/mt),Urea ($/mt),Potassium chloride ** ($/mt),Aluminum ($/mt),"Iron ore, cfr spot ($/dmtu)",Copper ($/mt),Lead ($/mt),Tin ($/mt),Nickel ($/mt),Zinc ($/mt),Gold ($/troy oz),Platinum ($/troy oz),Silver ($/troy oz)
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,i64,i64,f64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64,f64
"""1960M01""",1.6,1.6,1.6,null,null,null,0.1,0.4,null,null,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,null,null,334,233,null,94,204,92,null,null,20.4,45.0,39.0,104.5,null,null,null,null,59.9,null,0.14,0.12,0.71,0.3,null,1.43,0.12,0.12,0.07,1737.0,null,31.9,null,149.2,null,0.65,null,0.82,13.0,null,53.0,42.3,28.5,511,11.4,715,206,2180,1631,261,35,84,0.9
"""1960M02""",1.6,1.6,1.6,null,null,null,0.1,0.4,null,null,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,null,null,341,229,null,91,201,87,null,null,20.4,44.0,39.0,103.5,null,null,null,null,61.0,null,0.14,0.11,0.71,0.3,null,1.5,0.12,0.12,0.07,1737.0,null,31.9,null,149.2,null,0.65,null,0.83,13.0,null,53.0,42.3,28.5,511,11.4,728,204,2180,1631,245,35,84,0.9
"""1960M03""",1.6,1.6,1.6,null,null,null,0.1,0.4,null,null,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,null,null,338,225,null,92,201,84,null,null,20.7,45.0,35.0,103.8,null,null,null,null,61.7,null,0.14,0.13,0.77,0.3,null,1.5,0.12,0.12,0.07,1737.0,null,31.9,null,149.2,null,0.65,null,0.86,13.0,null,53.0,42.3,28.5,511,11.4,685,210,2174,1631,249,35,84,0.9
"""1960M04""",1.6,1.6,1.6,null,null,null,0.1,0.4,null,null,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,null,null,333,225,null,93,207,87,null,null,20.6,45.0,35.0,101.0,null,null,null,null,61.0,null,0.14,0.14,0.84,0.3,null,1.68,0.12,0.12,0.07,1737.0,null,31.9,null,149.2,null,0.64,null,0.86,13.0,null,53.0,42.3,28.5,511,11.4,723,214,2178,1631,255,35,84,0.9
"""1960M05""",1.6,1.6,1.6,null,null,null,0.1,0.4,null,null,0.6,0.92,0.69,1.03,0.93,1.12,1.04,321,null,null,335,225,null,93,209,82,null,null,20.6,48.0,35.0,102.2,null,null,null,null,57.7,null,0.15,0.14,0.76,0.3,null,1.76,0.12,0.12,0.07,1737.0,null,31.9,null,149.2,null,0.65,null,0.93,13.0,null,53.0,42.3,28.5,511,11.4,685,213,2163,1631,254,35,84,0.9


## 4. Spot-check the 90-table profiling shortlist

03's shortlist findings (constant columns, outlier columns) were only verified in
detail for a couple of example tables. Spot-check two more from the shortlist here —
not exhaustive, just enough to catch anything that doesn't match the earlier pattern
before 05 treats the whole shortlist as "understood."

In [19]:
# Adjust these two table names based on what stands out in 03's profiling_shortlist —
# defaulting to two with the highest constant_columns count as a starting point.
spotcheck_tables = ["asti_researchers", "asti_expenditures"]

for table in spotcheck_tables:
    print(f"--- {table}: sample rows ---")
    sample = conn.execute(f"SELECT * FROM raw.{table} LIMIT 5").pl()
    print(sample)
    print()

--- asti_researchers: sample rows ---
shape: (5, 18)
┌───────────┬───────────┬─────────┬───────────┬──────────────────┬─────────────┬────────┬──────────┬──────────┬───────┬─────────────┬──────────────────┬───────────┬──────┬──────┬───────┬──────┬──────┐
│ Area Code ┆ Area Code ┆ Area    ┆ Indicator ┆ Indicator        ┆ Degree Code ┆ Degree ┆ Sex Code ┆ Sex Code ┆ Sex   ┆ Institution ┆ Institution      ┆ Year Code ┆ Year ┆ Unit ┆ Value ┆ Flag ┆ Note │
│ ---       ┆ (M49)     ┆ ---     ┆ Code      ┆ ---              ┆ ---         ┆ ---    ┆ ---      ┆ (SDG)    ┆ ---   ┆ Code        ┆ ---              ┆ ---       ┆ ---  ┆ ---  ┆ ---   ┆ ---  ┆ ---  │
│ i64       ┆ ---       ┆ str     ┆ ---       ┆ str              ┆ i64         ┆ str    ┆ i64      ┆ ---      ┆ str   ┆ ---         ┆ str              ┆ i64       ┆ i64  ┆ str  ┆ f64   ┆ str  ┆ str  │
│           ┆ str       ┆         ┆ i64       ┆                  ┆             ┆        ┆          ┆ str      ┆       ┆ i64         ┆          

### Notes — Raw Data Exploration

- **Resolved — Q1 (areacodes duplicates):** `foodbalancesheets_areacodes` (213/426) and
  `foodbalancesheetshistoric_areacodes` (217/434) are exact 2x duplicates — every row
  appears exactly twice. `forestry_trade_flows_areacodes` (216 distinct / 423 total, not a
  clean 2x split) is also exact duplication, just not uniformly 2x per row. **Action for 05:**
  plain `DISTINCT` in staging resolves all 3 — no normalization needed.
- **Resolved — Q2 (trade_matrix):** flag values are `A`, `E`, `I`, `X` (standard FAOSTAT
  codes — confirm exact meaning against the `raw.trade_flags` lookup table when writing the
  staging model). 0.00% of rows are null across every year (0 / 6,640,547) — every row has
  at least one real value somewhere in its 39-year span. **Action for 05:** unpivot to long
  format (`reporter, partner, item, element, year, value, flag`); no pre-filtering of
  all-null rows needed.
- **Resolved — Q3 (commodity_prices):** NOT wide-by-year like `trade_matrix` — it's
  wide-by-commodity. One `Date` column (monthly, e.g. `1960M01`) plus 71 columns, each a
  distinct commodity price series (crude oil variants, coal, gas, cocoa, coffee, grains,
  metals, etc.). The 2485 correlation pairs reflect genuine economic co-movement between
  commodities (e.g. crude oil benchmarks moving together), not redundant data. Also noted:
  some columns (`Coconut oil`, `Palm oil`, `Soybeans`, `Aluminum`, `Copper`, etc.) are typed
  `BIGINT` while sibling columns are `DOUBLE` — inconsistent, likely just an artifact of
  those series having no decimals in the inferred sample. **Action for 05:** unpivot to long
  format (`date, commodity, price`); explicitly cast all price columns to `DOUBLE` in staging
  to avoid type mismatches after unpivoting.
- **Resolved — Q4 (spotcheck):** `asti_researchers` and `asti_expenditures` matched 03's
  prediction exactly — standard long/normalized FAOSTAT tables (`Area`, `Indicator`, `Value`,
  `Flag`, `Note`). Their "constant columns" (e.g. `Cost Category Code` always `300` = "Total",
  `Sex Code` mostly `1` = "Total") are legitimate filter artifacts — these datasets appear to
  only have aggregated "Total" rows loaded, not broken out by sub-category. **Action for 05:**
  no surprises; 03's shortlist can be treated as reliable without further spot-checking.
- All four questions resolved — **04 is complete.** Carry this table's findings directly into
  **05 — Data Cleaning Plan** as concrete dbt staging decisions.

In [20]:
conn.close()
print("Connection closed")

Connection closed
